# 03 - The daily engine's arithmetic

Milestone 3 of `vol-strategy-backtester` runs a backtest one session at a time: it fills orders, settles expiries, marks the book, hedges with SPY shares and writes a ledger. Every price and every cash flow in that loop is an integer number of cents, so the cash balance reconciles exactly. This notebook copies the three modules that hold that arithmetic out of `src/volbt/engine/` (`cash.py`, `fills.py`, `hedge.py`) and runs them on the ADRs' own worked numbers. It needs no market data, no store and no `volbt`.

This notebook is generated by `tools/build_notebooks.py`. Do not edit it by hand; edit `src/` and rebuild. The walkthrough of the whole engine, with its measured results, is in `docs/milestones/03-daily-engine.md`.

## Cash in integer cents

Float cash cannot reconcile: a sum of a few thousand float flows differs from the balance in its last bits. In cents every flow is an exact integer, rounded once, where it is created, with `Decimal` and `ROUND_HALF_UP` (never Python's `round()`). The functions are interest (ADR-0007), dividends and the borrow fee on short shares (ADR-0005). Copied from `src/volbt/engine/cash.py`.

In [ ]:
import math
from decimal import ROUND_HALF_UP, Decimal

In [ ]:
CENT = Decimal("0.01")
DAYS_PER_YEAR = 365

In [ ]:
def cents(amount: Decimal) -> int:
    """USD as a Decimal -> whole cents, ROUND_HALF_UP (6.175 -> 618, -6.175 -> -618)."""
    return int((Decimal(amount) / CENT).quantize(Decimal(1), rounding=ROUND_HALF_UP))

In [ ]:
def growth(rate: float, days: int) -> Decimal:
    """e^(rate days / 365) - 1, exactly the float expm1 gives, as a Decimal (no second rounding)."""
    return Decimal(math.expm1(rate * days / DAYS_PER_YEAR))

In [ ]:
def interest_cents(balance_cents: int, rate: float, days: int, negative_spread: Decimal) -> int:
    """ADR-0007 test 1: $1,000,000 at r_cc 5.2159% over 3 days earns $428.80; -$100,000 pays $46.99."""
    if days <= 0 or balance_cents == 0:
        return 0
    rate_used = rate + (float(negative_spread) if balance_cents < 0 else 0.0)
    return cents(Decimal(balance_cents) / 100 * growth(rate_used, days))

In [ ]:
def dividend_cents(shares: int, amount: float) -> int:
    """Signed: + for long shares, - for short. `amount` as stored (3 decimals), read exactly via its repr."""
    return cents(Decimal(shares) * Decimal(repr(amount)))

In [ ]:
def borrow_fee_cents(shares: int, price_cents: int, days: int, fee: Decimal) -> int:
    """The fee (a NEGATIVE flow) on short shares; 0 when the prior close was flat or long."""
    if shares >= 0 or days <= 0:
        return 0
    value = Decimal(-shares) * Decimal(price_cents) / 100
    return -cents(value * growth(float(fee), days))

In [ ]:
# ADR-0007 test 1: DGS3MO 5.25 is r_cc = 5.2159% (continuous): 365/91 ln(1 + 0.0525 x 91/365).
r_cc = (365 / 91) * math.log1p(5.25 / 100 * 91 / 365)
spread = Decimal('0.0050')   # a negative balance pays the rate plus 0.50%
print('r_cc:', round(r_cc * 100, 4), '%')
print(f'$1,000,000 over 3 days earns ${interest_cents(100_000_000, r_cc, 3, spread) / 100:,.2f}')
print(f'-$100,000 over 3 days pays ${-interest_cents(-10_000_000, r_cc, 3, spread) / 100:,.2f}')
# Rounding: 6.175 dollars is 618 cents half up; the float route gives 6.17.
print(cents(Decimal('6.175')), cents(Decimal('-6.175')), round(6.175, 2))
# ADR-0005: a short share pays the dividend; 1,000 shares short at $400.00 pay 0.25% a year to borrow.
print(dividend_cents(-100, 1.505), dividend_cents(5, 1.505))
print(borrow_fee_cents(-1000, 40_000, 3, Decimal('0.0025')))

## The fill price in integer cents

A fill is the mid plus or minus k half-spreads, rounded outward to the cent: a buy rounds up and a sell rounds down. ADR-0002 §2 requires integer arithmetic, because in float dollars `(2.00 + 2.10) / 2 - (2.10 - 2.00) / 2` is 1.9999999999999998 and a floor gives 1.99. The same module holds the quote states (valid, zero bid, no quote), the 25% spread rule for opening orders, the liquidity cap, the package rule and the costs. Copied from `src/volbt/engine/fills.py`, without its `LegFill` dataclass (a row type the notebook does not use). The module imports `cents` from `cash.py` (defined above) and five quality-flag constants from `volbt.data.flags` (notebook 01); the next cell gives it those five, and `tests/test_notebooks.py` checks they equal the real ones.

In [ ]:
from types import SimpleNamespace
quality = SimpleNamespace(BID_NULL=1 << 0, ASK_NULL=1 << 1, ZERO_BID=1 << 2, CROSSED=1 << 3, STALE=1 << 6)

In [ ]:
import math
from dataclasses import dataclass
from decimal import ROUND_CEILING, ROUND_FLOOR, Decimal
from fractions import Fraction

import numpy as np

In [ ]:
VALID, ZERO_BID, NO_QUOTE = "valid", "zero_bid", "no_quote"
NOT_VALID = quality.BID_NULL | quality.ASK_NULL | quality.CROSSED | quality.STALE | quality.ZERO_BID

In [ ]:
def price_cents(bid_c: int, ask_c: int, k: Fraction, side: int) -> int:
    """ADR-0002 §2 in integer cents: side +1 buys, -1 sells. k must be a multiple of 1/4."""
    n = k * 4
    if n.denominator != 1:
        raise ValueError(f"k = {k} is not a multiple of 1/4")
    n = int(n)
    if side > 0:
        return -(-(4 * (bid_c + ask_c) + n * (ask_c - bid_c)) // 8)
    return (4 * (bid_c + ask_c) - n * (ask_c - bid_c)) // 8

In [ ]:
def to_cents(price) -> int | None:
    """A vendor price in dollars (every one is a whole number of cents) as int cents; None for null/NaN."""
    if price is None or not np.isfinite(price):
        return None
    return int(round(float(price) * 100))

In [ ]:
def quote_state(bid_c: int | None, ask_c: int | None, quality_flags: int | None) -> str:
    """valid, zero_bid or no_quote (module docstring). A missing contract has no cents and no flags."""
    if bid_c is None or ask_c is None or quality_flags is None:
        return NO_QUOTE
    if ask_c <= 0 or bid_c > ask_c or (quality_flags & (quality.CROSSED | quality.STALE)):
        return NO_QUOTE
    if bid_c == 0:
        return ZERO_BID
    return VALID

In [ ]:
def opening_liquid(bid_c: int, ask_c: int, cfg) -> bool:
    """ADR-0002 §7: bid >= open_min_bid and (a - b) / mid <= 25%, in integers: (a - b) <= rel (a + b) / 2."""
    rel = cfg.open_max_rel_spread
    return bid_c >= cfg.open_min_bid_cents and 2 * (ask_c - bid_c) * rel.denominator <= rel.numerator * (ask_c + bid_c)

In [ ]:
def volume_reliable(cfg, created, volume) -> bool:
    return (cfg.data_source == "optionsdx" and created >= cfg.volume_reliable_from and volume is not None
            and np.isfinite(volume))

In [ ]:
def leg_cap(cfg, reduce_only: bool, cap_volume, created) -> tuple[int, str, bool]:
    """(contracts the leg may fill, cap_type, volume_reliable) for one order leg (§5)."""
    reliable = volume_reliable(cfg, created, cap_volume)
    if reduce_only or cfg.cap_mode != "volume" or not reliable:
        return cfg.max_contracts_per_line, "hard", reliable
    p = cfg.participation_rate
    return min(cfg.max_contracts_per_line, (int(cap_volume) * p.numerator) // p.denominator), "volume", reliable

In [ ]:
def package_count(requested: int, caps, ratios) -> int:
    """Whole packages an opening order fills: min(requested, min over legs floor(cap / |ratio|)).

    ADR-0002 test 5: a straddle with caps 85 and 60 fills 60 packages.
    """
    return min([requested] + [cap // abs(ratio) for cap, ratio in zip(caps, ratios)])

In [ ]:
def option_fee_cents(contracts: int, cfg) -> int:
    """ADR-0002 §3: 70 cents per contract per side; 120 contracts pay $84.00."""
    return abs(contracts) * cfg.fee_per_contract_cents

In [ ]:
def stock_cost_cents(shares: int, cfg) -> tuple[int, int]:
    """(commission, slippage) in cents for one SPY share order (ADR-0002 §4).

    Commission $0.005 a share with a $1.00 minimum per order, slippage $0.005 a share, each exact in
    Decimal and rounded ROUND_HALF_UP: 1,235 shares cost 618 + 618 cents (round() on floats gives 6.17).
    No order, no cost.
    """
    if shares == 0:
        return 0, 0
    size = Decimal(abs(shares))
    commission = max(size * cfg.stock_commission_per_share, cfg.stock_commission_min)
    return cents(commission), cents(size * cfg.stock_half_spread)

In [ ]:
def outward_cents(price: float, side: int) -> int:
    """A model price rounded outward to the tick: up for a buy, down for a sell."""
    value = Decimal(repr(float(price))) * 100
    return int(value.to_integral_value(rounding=ROUND_CEILING if side > 0 else ROUND_FLOOR))

In [ ]:
H_MODEL_OTHER_EXPIRY = 1 << 4   # fills.fill_flags: h_model came from another expiry (ADR-0002 §7), above SURFACE_* 1-8

In [ ]:
def h_model(day_quotes: dict | None, expiry, k: float, band: float) -> tuple[float | None, bool]:
    """ADR-0002 §7: the median half-spread of valid quotes in the same expiry within +-band log-moneyness.

    `day_quotes` maps expiry -> (log_moneyness array, half_spread array) of the day's valid quotes. If the
    contract's own expiry has none in the band, the nearest listed expiry (by days) that has some is used.
    Returns (h, True when it came from another expiry); (None, False) when no expiry has any.
    """
    if not day_quotes:
        return None, False

    def band_median(key):
        if key not in day_quotes:
            return None
        km, half = day_quotes[key]
        inside = np.abs(km - k) <= band
        return float(np.median(half[inside])) if inside.any() else None

    own = band_median(expiry)
    if own is not None:
        return own, False
    for key in sorted(day_quotes, key=lambda e: (abs((e - expiry).days), e)):
        found = band_median(key)
        if found is not None:
            return found, key != expiry
    return None, False

In [ ]:
def leg_price(state: str, reduce_only: bool, side: int, bid_c, ask_c, model_mid, h, cfg, surface_mode: bool):
    """(price_cents, fill_source, reason) for one leg's quote state, or (None, None, reason)."""
    if surface_mode or (reduce_only and state == NO_QUOTE):
        if model_mid is None or h is None or not np.isfinite(model_mid):
            return None, None, "NO_QUOTE"
        spread = float(cfg.surface_half_spread_mult) * h
        price = outward_cents(model_mid + side * spread, side)
        if price < 1:
            return None, None, "NO_QUOTE"
        return price, "surface", None
    if state == NO_QUOTE:
        return None, None, "NO_QUOTE"
    if state == ZERO_BID:
        if reduce_only and side > 0:
            return ask_c, "quote", None          # the mid of 0/ask is meaningless: pay the ask, whatever k is
        return None, None, "ZERO_BID"
    if not reduce_only and not opening_liquid(bid_c, ask_c, cfg):
        return None, None, "ILLIQUID"
    return price_cents(bid_c, ask_c, cfg.k, side), "quote", None

The fill functions read their parameters from the run's configuration. This cell holds the defaults of `RunConfig` (`src/volbt/engine/run_config.py`) that these functions use; `tests/test_notebooks.py` checks them against it.

In [ ]:
from datetime import date
cfg = SimpleNamespace(
    k=Fraction(1), fee_per_contract_cents=70,
    stock_commission_per_share=Decimal('0.005'), stock_commission_min=Decimal('1.00'),
    stock_half_spread=Decimal('0.005'),
    participation_rate=Fraction(1, 10), max_contracts_per_line=100, cap_mode='hard_only',
    data_source='optionsdx', volume_reliable_from=date(2021, 1, 4),
    open_min_bid_cents=5, open_max_rel_spread=Fraction(1, 4), surface_half_spread_mult=Fraction(1),
    hedge_mode='time', hedge_every_n=1, hedge_band_pct_nav=None)

In [ ]:
# ADR-0002 test 1: bid 2.00 / ask 2.10, in cents (200 and 210).
for k in (Fraction(1), Fraction(1, 2), Fraction(0)):
    buy, sell = price_cents(200, 210, k, +1), price_cents(200, 210, k, -1)
    print(f'k = {k}: buy {buy / 100:.2f}  sell {sell / 100:.2f}')
# ADR-0002 test 8: the float-dollar form of the k = 1 sell is not 2.00, and a floor of it is 1.99.
dollars = (2.00 + 2.10) / 2 - (2.10 - 2.00) / 2
print(repr(dollars), math.floor(dollars * 100) / 100)

In [ ]:
# ADR-0002 §7: an opening order needs a bid of 5 cents and a spread of at most 25% of the mid.
print(opening_liquid(70, 90, cfg), opening_liquid(70, 91, cfg), opening_liquid(4, 5, cfg))
# 0.70 / 0.90 is exactly 25%. In float dollars it falls on the wrong side of the line:
print((0.90 - 0.70) / ((0.90 + 0.70) / 2) > 0.25)
# ADR-0002 test 2: a zero bid. A buy to close pays the ask whatever k is; a sale to open is refused.
state = quote_state(0, 5, 0)
print(state, leg_price(state, True, +1, 0, 5, None, None, cfg, False),
      leg_price(state, False, -1, 0, 5, None, None, cfg, False))

In [ ]:
# ADR-0002 tests 4 and 5. The diagnostic volume cap is 10% of the order date's volume:
# volumes 850 and 600 give caps 85 and 60, so a straddle of 120 packages fills 60.
volume_cfg = SimpleNamespace(**{**vars(cfg), 'cap_mode': 'volume'})
order_day = date(2022, 3, 14)
call_cap, put_cap = (leg_cap(volume_cfg, False, v, order_day)[0] for v in (850, 600))
n = package_count(120, [call_cap, put_cap], [-1, -1])
print(call_cap, put_cap, n, f'fees ${option_fee_cents(2 * n, cfg) / 100:.2f}')

In [ ]:
# ADR-0002 test 6: 1,235 shares: commission and slippage are each 1,235 x 0.005 = 6.175 exactly.
print(stock_cost_cents(1235, cfg), 'cents; round() on floats gives', round(1235 * 0.005, 2))
# The $1.00 minimum commission applies to 100 shares; no order, no cost.
print(stock_cost_cents(100, cfg), stock_cost_cents(0, cfg))

## The hedge trade

Net delta is the sum over the open lots of quantity x 100 x delta, plus the shares held. The hedge trades the negative of it, rounded to a whole share with halves away from zero: Python's `round()` rounds halves to the even number, so `round(1234.5)` is 1234. A hedge is due on a schedule (every n-th session), or when the exposure is outside a band of NAV, or never. Copied from `src/volbt/engine/hedge.py`.

In [ ]:
from decimal import ROUND_HALF_UP, Decimal

In [ ]:
def round_half_away(value: float) -> int:
    """Nearest integer, halves away from zero: 1234.5 -> 1235, -1234.5 -> -1235, 2.4999 -> 2."""
    return int(Decimal(repr(float(value))).quantize(Decimal(1), rounding=ROUND_HALF_UP))

In [ ]:
def hedge_shares(net_delta_shares: float) -> int:
    """The shares to trade to bring the book back to zero delta."""
    return -round_half_away(net_delta_shares)

In [ ]:
def hedge_due(cfg, session_index: int, net_delta_shares: float, spot: float, nav_mid: float) -> tuple[bool, str]:
    """(due, trigger). session_index counts the run's sessions from 0.

    time:  every n-th session from the first: n = 5 hedges on sessions 1, 6, 11 (indices 0, 5, 10).
    band:  checked daily, due when |net delta x S| > band x NAV_mid (ADR-0005 §4).
    never: not due (an unhedged book; ADR-0003 §3 liquidates delivered shares instead).
    """
    if cfg.hedge_mode == "never":
        return False, "none"
    if cfg.hedge_mode == "time":
        return session_index % cfg.hedge_every_n == 0, "schedule"
    exposure = Decimal(repr(abs(net_delta_shares * spot)))
    return exposure > Decimal(repr(nav_mid)) * Decimal(cfg.hedge_band_pct_nav.numerator) / cfg.hedge_band_pct_nav.denominator, "band"

In [ ]:
# ADR-0005 test 2: a net delta of +1,234.5 shares sells 1,235.
print(round_half_away(1234.5), round(1234.5), hedge_shares(1234.5), hedge_shares(-1234.5))
shares = hedge_shares(1234.5)
print('trade', shares, 'shares; (commission, slippage) in cents:', stock_cost_cents(shares, cfg))
# ADR-0005 test 3: a 1% band at NAV $1,000,000 and S = 400 is 25 shares.
band = SimpleNamespace(**{**vars(cfg), 'hedge_mode': 'band', 'hedge_band_pct_nav': Fraction(1, 100)})
print(hedge_due(band, 0, 20.0, 400.0, 1_000_000.0), hedge_due(band, 0, 30.0, 400.0, 1_000_000.0),
      hedge_shares(30.0))
# Every 5th session, counted from the first: indices 0, 5 and 10 (sessions 1, 6 and 11).
every_5 = SimpleNamespace(**{**vars(cfg), 'hedge_every_n': 5})
print([i for i in range(11) if hedge_due(every_5, i, 10.0, 400.0, 1e6)[0]])